In [ ]:
from pathlib import Path

In [ ]:
from sj_ai_utils.datasets.esic_v1 import ESICv1Dataset
from sj_ai_utils.datasets.l_hotse import AMI, VoxPopuli, LibriSpeech, Tedlium

In [ ]:
from rt_whisper_optimizer import Optimizer

In [ ]:
ESIC_TRAIN = "/workspaces/dev/test/performance_test/esic/data/train.json"
AMI_PATH = "/workspaces/dev/.datasets/ami"
VOX_POPULI_PATH = "/workspaces/dev/.datasets/vox_populi"
LIBRI_PATH = "/workspaces/dev/.datasets/libri_speech"
TEDLIUM_PATH = "/workspaces/dev/.datasets/tedlium"

STORAGE = "/workspaces/dev/.storage/all/"
STUDY = "/workspaces/dev/test/optimize/all/study/"
CACHE = "/workspaces/dev/test/optimize/all/.cache/esic_ami_vox_libri_tedlium.pkl"

In [ ]:
OUTPUT = [
    "/workspaces/dev/test/optimize/all/hyperparameters/20250826/step1_16b-96k",
    "/workspaces/dev/test/optimize/all/hyperparameters/20250826/step1_16b-96k-16cs",
    "/workspaces/dev/test/optimize/all/hyperparameters/20250826/step1_16b-112k",
    "/workspaces/dev/test/optimize/all/hyperparameters/20250826/step1_16b-112k-16cs",
]
STUDY_INSTRUCTION = [
    "/workspaces/dev/test/optimize/all/study_param/20250826/step1_16b-96k.yaml",
    "/workspaces/dev/test/optimize/all/study_param/20250826/step1_16b-96k-16cs.yaml",
    "/workspaces/dev/test/optimize/all/study_param/20250826/step1_16b-112k.yaml",
    "/workspaces/dev/test/optimize/all/study_param/20250826/step1_16b-112k-16cs.yaml",
]

In [ ]:
esic_train = Path(ESIC_TRAIN)
ami_path = Path(AMI_PATH)
vox_populi_path = Path(VOX_POPULI_PATH)
libri_speech_path = Path(LIBRI_PATH)
tedlium_path = Path(TEDLIUM_PATH)

storage = Path(STORAGE)
study = Path(STUDY)
cache = Path(CACHE)
if not esic_train.exists():
    raise FileNotFoundError(f"Train file not found: {esic_train}")

In [ ]:
esic = ESICv1Dataset.load(esic_train)
ami = AMI(ami_path)
ami_ihm = ami.load_dev_ihm()[:10]
ami_sdm = ami.load_dev_sdm()[:10]
vox_populi = VoxPopuli(vox_populi_path).load_dev_asr_en()[:10]
libri_speech = LibriSpeech(libri_speech_path).load_dev_clean()[:100]
tedlium = Tedlium(tedlium_path).load_dev()[:10]

In [ ]:
datasets = esic + ami_ihm + ami_sdm + vox_populi + libri_speech + tedlium

In [ ]:
for output_path, study_instruction in zip(OUTPUT, STUDY_INSTRUCTION):
    output = Path(output_path)
    study_instruction = Path(study_instruction)
    esic_optimizer = Optimizer(datasets, study, output, param_cache_path=cache, cache_storage=storage)
    esic_optimizer.optimize(study_instruction)